# Phase 6 - Multimodal Fusion (XGBoost)

This phase creates **one row per child** from Phase 1-5 outputs and predicts four screening risk indicators: Social Communication, Attention Regulation, Speech & Language, and Repetitive Behaviour.

> **Clinical safety:** This is screening decision support only, not a diagnosis. Train only on de-identified, ethically approved data with clinician labels; validate it before clinical use.

## 1. Install dependencies
Run once in Colab (or install `requirements-phase6.txt` locally).

In [ ]:
!pip install -q -r requirements-phase6.txt

## 2. Input contract

- Run Phase 4 batch mode over **all** `.mp3`, `.wav`, or `.mp4` recordings for a child. It appends one row per recording to `audio_features.csv`.
- Run Phase 5 once per child to make `questionnaire_features.csv`.
- Export numeric Phase 1-3 results to CSVs with a `child_id` column. Multiple videos, frames, or windows are allowed.
- The supplied `.pth` files alone are not enough to run raw-video inference: their model architectures, preprocessing transforms, and label maps are not in this folder. Do not infer them from checkpoint names.

In [ ]:
from pathlib import Path
import pandas as pd
from phase6_fusion import build_fusion_row, train_models, predict

CHILD_ID = 'child_001'
# Set a path to None only when that modality is unavailable.
PHASE1_CSV = Path('phase1_features.csv')  # e.g. emotion probabilities per video/frame
PHASE2_CSV = Path('phase2_features.csv')  # e.g. face/attention measures
PHASE3_CSV = Path('phase3_features.csv')  # e.g. repetitive-behaviour probabilities
AUDIO_CSV = Path('audio_features.csv')
QUESTIONNAIRE_CSV = Path('questionnaire_features.csv')

def existing(path):
    return path if path is not None and path.exists() else None

fusion_row = build_fusion_row(
    CHILD_ID,
    phase1_csv=existing(PHASE1_CSV), phase2_csv=existing(PHASE2_CSV),
    phase3_csv=existing(PHASE3_CSV), audio_csv=existing(AUDIO_CSV),
    questionnaire_csv=existing(QUESTIONNAIRE_CSV),
)
fusion_row.to_csv(f'{CHILD_ID}_fusion_features.csv', index=False)
display(fusion_row.T)

## 3. Build training data

Concatenate one fusion row per **different child** into `fusion_training.csv`, then add the four clinician-provided labels below. Do not manufacture labels from M-CHAT-R or use the same child in train and test.

In [ ]:
LABEL_COLUMNS = [
    'y_social_communication', 'y_attention_regulation',
    'y_speech_language', 'y_repetitive_behaviour',
]
# Example only: after creating one row per labelled child.
# training = pd.concat([pd.read_csv(p) for p in Path('.').glob('*_fusion_features.csv')], ignore_index=True)
# training = training.merge(clinician_labels[['child_id', *LABEL_COLUMNS]], on='child_id', validate='one_to_one')
# training.to_csv('fusion_training.csv', index=False)

## 4. Train four XGBoost models
The module uses a stratified held-out split and saves the feature schema, metrics, and one model for each risk category. It will stop if labels are missing or a target contains only one class.

In [ ]:
# Uncomment only when a real, labelled fusion_training.csv exists.
# metrics = train_models('fusion_training.csv', 'phase6_models')
# metrics

## 5. Predict one child

Each XGBoost score is blended with the matching Phase-5 questionnaire axis (60% model / 40% questionnaire). An M-CHAT-R total of 8+ sets the referral override, independently of model output.

In [ ]:
# Uncomment after training has created phase6_models/.
# results = predict(fusion_row, 'phase6_models', threshold=0.50)
# results.to_csv(f'{CHILD_ID}_phase6_screening.csv', index=False)
# display(results.T)

## Deployment recommendation
For a free prototype, choose **Streamlit Community Cloud** with a Streamlit UI that uploads files, runs the light CPU feature path, and keeps no uploads. Do not deploy identifiable child video/audio or represent the output as medical diagnosis on a public free service. The three checkpoints and Whisper/video analysis may exceed free CPU memory or startup limits; for the complete raw-media pipeline, use a private, properly secured paid deployment after validation.